In [ ]:
# ============================================================
# CELL 1 — Setup. Run this first, every session.
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

!pip install ultralytics -q

import os, cv2, glob, torch, json, datetime
import torch.nn as nn
from ultralytics import YOLO
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

BASE = "/content/drive/MyDrive/mess_project"
device = "cuda" if torch.cuda.is_available() else "cpu"
detection_model = YOLO("yolo11x.pt")

print("Setup complete")

In [ ]:
# ============================================================
# CELL 2 — Incremental crop extraction. Safe to re-run anytime.
# Only crops NEW photos added to mess_photos/, never re-crops
# photos already processed (tracked via manifest file).
# ============================================================

def extract_crops_incremental(image_dir, output_dir, manifest_path, conf_threshold=0.065):
    os.makedirs(output_dir, exist_ok=True)

    if os.path.exists(manifest_path):
        with open(manifest_path) as f:
            processed = set(json.load(f))
    else:
        processed = set()

    all_images = (
        glob.glob(f"{image_dir}/*.jpg")
        + glob.glob(f"{image_dir}/*.jpeg")
        + glob.glob(f"{image_dir}/*.png")
    )
    new_images = [p for p in all_images if os.path.basename(p) not in processed]

    print(f"{len(all_images)} total source photos, {len(processed)} already processed, {len(new_images)} new")

    if not new_images:
        print("Nothing new to crop.")
    else:
        crop_id = 0
        tag = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        for img_path in new_images:
            results = detection_model(img_path, conf=conf_threshold, classes=[0])[0]
            img = cv2.imread(img_path)
            for box in results.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                crop = img[max(0, y1):y2, max(0, x1):x2]
                if crop.size > 0:
                    cv2.imwrite(f"{output_dir}/{tag}_crop_{crop_id}.jpg", crop)
                    crop_id += 1
            processed.add(os.path.basename(img_path))

        with open(manifest_path, "w") as f:
            json.dump(sorted(processed), f, indent=2)

        print(f"Extracted {crop_id} new crops to {output_dir}")
        print(f"Manifest updated: {len(processed)} source photos now marked processed")


manifest_path = f"{BASE}/processed_photos.json"
raw_crops_dir = f"{BASE}/raw_crops"
extract_crops_incremental(f"{BASE}/mess_photos", raw_crops_dir, manifest_path)


In [ ]:

# ============================================================
# CELL 3 — Check how many crops still need sorting.
# Points at the SAME stable raw_crops_dir from Cell 2.
# ============================================================
raw_crops_files = set(
    f for f in os.listdir(raw_crops_dir) if f.endswith(('.jpg', '.jpeg', '.png'))
)

already_sorted = set()
for folder in [
    f"{BASE}/dataset/queue_classifier/queue",
    f"{BASE}/dataset/queue_classifier/not_queue",
    f"{BASE}/dataset/seated_classifier/seated",
    f"{BASE}/dataset/seated_classifier/standing",
]:
    os.makedirs(folder, exist_ok=True)
    already_sorted.update(f for f in os.listdir(folder) if f.endswith(('.jpg', '.jpeg', '.png')))

unsorted = raw_crops_files - already_sorted
print(f"Total in raw_crops: {len(raw_crops_files)}")
print(f"Already sorted (by filename): {len(already_sorted)}")
print(f"Need sorting: {len(unsorted)}")
print(sorted(unsorted))


In [ ]:

# ============================================================
# CELL 4 — Labeling widget. Defines functions only, labels nothing yet.
# ============================================================
!pip install ipywidgets -q
import ipywidgets as widgets
from IPython.display import display, clear_output
import shutil

def label_crops_simplified(source_dir, folders, base_out_dir):
    """
    One click per crop: "Queue" (also marks standing), "Seated" (also
    marks not_queue), or "Other/skip" for anyone who doesn't clearly
    fit either (e.g. staff, someone just walking through) — skipping
    avoids polluting a class with a wrongly-forced label.
    """
    for folder in folders.values():
        os.makedirs(folder, exist_ok=True)

    already_sorted = set()
    for folder in folders.values():
        already_sorted.update(os.listdir(folder))

    all_crops = sorted(glob.glob(f"{source_dir}/*.jpg") + glob.glob(f"{source_dir}/*.png"))
    remaining = [p for p in all_crops if os.path.basename(p) not in already_sorted]

    print(f"{len(remaining)} crops left to label out of {len(all_crops)} total")

    state = {"idx": 0}
    out = widgets.Output()

    def show_next():
        with out:
            clear_output(wait=True)
            if state["idx"] >= len(remaining):
                print("Done — no more crops to label.")
                return
            path = remaining[state["idx"]]
            display(widgets.Image(value=open(path, "rb").read(), width=200))
            print(f"{state['idx']+1}/{len(remaining)}  —  {os.path.basename(path)}")

    def label_queue(_):
        if state["idx"] >= len(remaining): return
        path = remaining[state["idx"]]
        fname = os.path.basename(path)
        shutil.copy(path, os.path.join(folders["queue"], fname))
        shutil.copy(path, os.path.join(folders["standing"], fname))
        state["idx"] += 1
        show_next()

    def label_seated(_):
        if state["idx"] >= len(remaining): return
        path = remaining[state["idx"]]
        fname = os.path.basename(path)
        shutil.copy(path, os.path.join(folders["seated"], fname))
        shutil.copy(path, os.path.join(folders["not_queue"], fname))
        state["idx"] += 1
        show_next()

    def skip(_):
        state["idx"] += 1
        show_next()

    queue_btn = widgets.Button(description="Queue (standing in line)", button_style="danger")
    seated_btn = widgets.Button(description="Seated", button_style="info")
    skip_btn = widgets.Button(description="Other / skip", button_style="")

    queue_btn.on_click(label_queue)
    seated_btn.on_click(label_seated)
    skip_btn.on_click(skip)

    display(widgets.HBox([queue_btn, seated_btn, skip_btn]))
    display(out)
    show_next()

print("Labeling widget ready")



In [ ]:

# ============================================================
# CELL 5 — Run the labeler. Click through crops.
# ============================================================
label_crops_simplified(
    source_dir=raw_crops_dir,
    folders={
        "queue": f"{BASE}/dataset/queue_classifier/queue",
        "not_queue": f"{BASE}/dataset/queue_classifier/not_queue",
        "seated": f"{BASE}/dataset/seated_classifier/seated",
        "standing": f"{BASE}/dataset/seated_classifier/standing",
    },
    base_out_dir=BASE,
)

In [ ]:
######### not required now as i am doing on the basis ofo cell 5


def label_posture(source_dir, base=BASE):
    D = lambda p: f"{base}/dataset/{p}"
    dest = {
        "seated":     [D("seated_classifier/seated"),   D("queue_classifier/not_queue")],
        "stand_line": [D("seated_classifier/standing"), D("queue_classifier/queue")],
        "stand_free": [D("seated_classifier/standing"), D("queue_classifier/not_queue")],
    }
    done = set()
    for dirs in dest.values():
        for d in dirs:
            os.makedirs(d, exist_ok=True)
            done.update(os.listdir(d))
    todo = [p for p in sorted(glob.glob(f"{source_dir}/*.jpg") + glob.glob(f"{source_dir}/*.png"))
            if os.path.basename(p) not in done]
    print(f"{len(todo)} crops to label")

    state, out = {"i": 0}, widgets.Output()

    def show():
        with out:
            clear_output(wait=True)
            if state["i"] >= len(todo):
                print("Done.")
                return
            display(widgets.Image(value=open(todo[state["i"]], "rb").read(), width=220))
            print(f"{state['i'] + 1}/{len(todo)}")

    def handler_for(kind):
        def handler(_):
            if state["i"] >= len(todo):
                return
            p = todo[state["i"]]
            for d in dest.get(kind, []):
                shutil.copy(p, os.path.join(d, os.path.basename(p)))
            state["i"] += 1
            show()
        return handler

    specs = [("Seated", "seated", "info"), ("Standing - in line", "stand_line", "danger"),
             ("Standing - NOT in line", "stand_free", "warning"), ("Skip", "skip", "")]
    btns = []
    for label, kind, style in specs:
        b = widgets.Button(description=label, button_style=style)
        b.on_click(handler_for(kind))
        btns.append(b)
    display(widgets.HBox(btns))
    display(out)
    show()

# Put 10-15 NEW high-angle / overhead photos in this folder first
os.makedirs(f"{BASE}/train_photos_overhead", exist_ok=True)
extract_crops_incremental(f"{BASE}/train_photos_overhead", f"{BASE}/raw_crops_overhead",
                          f"{BASE}/processed_overhead.json", conf_threshold=0.25)
label_posture(f"{BASE}/raw_crops_overhead")

In [ ]:

# ============================================================
# CELL 6 — Train both classifiers. Run after labeling enough crops
# (aim for 30+ per class before trusting results).
# ============================================================
def train_classifier(data_dir, save_path, num_classes=2, epochs=15, batch_size=8, lr=0.001):
    transform = transforms.Compose([
        transforms.Resize((128, 128)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    dataset = datasets.ImageFolder(data_dir, transform=transform)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    model = models.mobilenet_v2(weights="IMAGENET1K_V1")
    model.classifier[1] = nn.Linear(model.last_channel, num_classes)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    for epoch in range(epochs):
        total_loss = 0
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}/{epochs} — loss: {total_loss/len(loader):.4f}")
    torch.save(model.state_dict(), save_path)
    print(f"Saved: {save_path}. Classes: {dataset.classes}")

train_classifier(f"{BASE}/dataset/queue_classifier", f"{BASE}/queue_classifier.pt")
train_classifier(f"{BASE}/dataset/seated_classifier", f"{BASE}/seated_classifier.pt")


In [ ]:

# ============================================================
# CELL 7 — Load classifiers + full pipeline.
# ============================================================
_transform = transforms.Compose([
    transforms.Resize((128, 128)), transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def load_classifier(weights_path, class_names):
    model = models.mobilenet_v2(weights=None)
    model.classifier[1] = nn.Linear(model.last_channel, len(class_names))
    model.load_state_dict(torch.load(weights_path, map_location=device))
    model.to(device).eval()
    return model

def classify_crop(crop_bgr, model, class_names):
    img_pil = Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))
    tensor = _transform(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(tensor).argmax(dim=1).item()
    return class_names[pred]

queue_model_loaded = load_classifier(f"{BASE}/queue_classifier.pt", ["not_queue", "queue"])
seated_model_loaded = load_classifier(f"{BASE}/seated_classifier.pt", ["seated", "standing"])

QUEUE_WEIGHT, SEATED_WEIGHT, HEADCOUNT_WEIGHT = 0.5, 0.2, 0.3
MAX_QUEUE, MAX_SEATED, MAX_HEADCOUNT = 30, 60, 80

def compute_crowd_score(queue_count, seated_count, total_headcount):
    q = min(queue_count / MAX_QUEUE, 1)
    s = min(seated_count / MAX_SEATED, 1)
    h = min(total_headcount / MAX_HEADCOUNT, 1)
    score = q * QUEUE_WEIGHT + s * SEATED_WEIGHT + h * HEADCOUNT_WEIGHT
    level = "green" if score < 0.4 else ("amber" if score < 0.7 else "red")
    return {"score": round(score, 3), "level": level}

DEFAULT_CONF = 0.15   # <- set this to whatever you've validated works
                      #    best on your real test images; just make it
                      #    ONE number both cells reference, not two.

# update run_pipeline's default to reference it:
def run_pipeline(image_path, conf=DEFAULT_CONF):
    img = cv2.imread(image_path)
    results = detection_model(image_path, conf=conf, iou=0.5, classes=[0], agnostic_nms=True)[0]
    detections = [{"bbox": b.xyxy[0].tolist(), "confidence": round(float(b.conf[0]), 3)} for b in results.boxes]
    queue_count = seated_count = standing_other_count = 0
    for det in detections:
        x1, y1, x2, y2 = map(int, det["bbox"])
        crop = img[max(0, y1):y2, max(0, x1):x2]
        if crop.size == 0:
            continue
        det["in_queue"] = classify_crop(crop, queue_model_loaded, ["not_queue", "queue"]) == "queue"
        det["seated"] = classify_crop(crop, seated_model_loaded, ["seated", "standing"]) == "seated"
        if det["in_queue"]:
            queue_count += 1
        elif det["seated"]:
            seated_count += 1
        else:
            standing_other_count += 1   # standing, but not classified as in queue
    total = len(detections)
    crowd = compute_crowd_score(queue_count, seated_count, total)
    return {
        "headcount": total,
        "queue_count": queue_count,
        "seated_count": seated_count,
        "standing_other_count": standing_other_count,  # NEW: makes the 3 buckets mutually exclusive and add up to headcount
        **crowd,
        "detections": detections,
    }

print("run_pipeline updated with consistent conf + mutually-exclusive counts")



In [ ]:
# ======================================================================
# CELL 7b  (NEW, paste right after Cell 7)  - gives each person a seated probability
# ======================================================================
def seated_probability(crop_bgr):
    """Probability (0-1) that this crop is a seated person, from your trained seated classifier."""
    t = _transform(Image.fromarray(cv2.cvtColor(crop_bgr, cv2.COLOR_BGR2RGB))).unsqueeze(0).to(device)
    with torch.no_grad():
        return torch.softmax(seated_model_loaded(t), dim=1)[0][0].item()   # index 0 = "seated"

print("seated_probability ready")

In [ ]:

# ======================================================================
# CELL 7c  (REPLACE)  - queue grouping + rescue of queue members hidden by tables
# ======================================================================
import math

def foot_point(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, y2)          # bottom-center = where the person stands

def box_height(bbox):
    return max(bbox[3] - bbox[1], 1)

def pair_distance(det_a, det_b, normalize=True):
    ax, ay = foot_point(det_a["bbox"])
    bx, by = foot_point(det_b["bbox"])
    d = math.hypot(ax - bx, ay - by)
    if normalize:
        # divide by avg person height so threshold = "person-heights", not pixels
        d = d / ((box_height(det_a["bbox"]) + box_height(det_b["bbox"])) / 2)
    return d

def point_in_polygon(pt, polygon):
    """Ray casting. polygon = [(x, y), ...] in original-image pixel coordinates."""
    x, y = pt
    inside = False
    n = len(polygon)
    for i in range(n):
        x1, y1 = polygon[i]
        x2, y2 = polygon[(i + 1) % n]
        if (y1 > y) != (y2 > y):
            x_cross = x1 + (y - y1) * (x2 - x1) / (y2 - y1)
            if x < x_cross:
                inside = not inside
    return inside

def group_elongation(points):
    """
    ~1.0 for a round blob (people clustered in a circle), large for a line.
    Ratio of the spread along the group's main axis vs the perpendicular axis.
    """
    n = len(points)
    mx = sum(p[0] for p in points) / n
    my = sum(p[1] for p in points) / n
    a = sum((p[0] - mx) ** 2 for p in points) / n
    c = sum((p[1] - my) ** 2 for p in points) / n
    b = sum((p[0] - mx) * (p[1] - my) for p in points) / n
    half = (a + c) / 2
    root = math.sqrt(((a - c) / 2) ** 2 + b * b)
    l1, l2 = half + root, half - root
    if l1 <= 1e-9:
        return 1.0
    return math.sqrt(l1 / max(l2, 1e-9))

def assign_queue_by_distance(detections, dist_threshold=1.0, min_group_size=3, normalize=True,
                             min_elongation=None, queue_zone=None, zone_min_fraction=0.5,
                             min_group_size_in_zone=2, min_classifier_fraction=None,
                             min_member_conf=None, rescue_max_p_seated=None, rescue_head_tol=0.2):
    """
    Step 1: link STANDING people whose feet are within dist_threshold (chained into groups).
    Step 2: keep a group as a queue only if it passes the filters:

      NO ZONE given:
        - at least min_group_size people (3 stops two friends chatting counting as a queue)
        - if min_elongation is set: the group must be line-shaped, not a round huddle
      ZONE given (polygon around the serving/queue area):
        - at least zone_min_fraction of the group must stand inside the zone
        - at least min_group_size_in_zone people (2 is fine here, a pair at the counter
          IS a queue). The shape check is skipped, a crowd at the counter still counts.
      EITHER way, if min_classifier_fraction is set: that fraction of the group must also
        have det["queue_vote"] == True (your trained queue classifier agreeing).

    min_member_conf: detections below this confidence never join a group (they still count in
      the headcount). Stops a shaky box from extending a line.
    rescue_max_p_seated: when a queue is found, someone the seated classifier called "seated" but
      with p_seated <= this value is moved INTO the queue if they stand in line with queue members
      (within dist_threshold AND head height within rescue_head_tol of that member's). This fixes
      queue members whose legs are hidden by tables: head+torso looks seated, but their position in
      the line says otherwise. None = off.

    Seated people are otherwise ignored. Sets det["in_queue"], det["queue_group"],
    det["queue_reject"], det["rescued"] in place. Returns (number_of_queue_groups, rejected) where rejected is a list of
    {"size": n, "reason": ...} for groups of 2+ people that were linked but NOT called a queue.
    """
    standing_idx = [i for i, d in enumerate(detections)
                    if not d.get("seated", False)
                    and (min_member_conf is None or d.get("confidence", 1.0) >= min_member_conf)]
    parent = {i: i for i in standing_idx}

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for a in range(len(standing_idx)):
        for b in range(a + 1, len(standing_idx)):
            i, j = standing_idx[a], standing_idx[b]
            if pair_distance(detections[i], detections[j], normalize) <= dist_threshold:
                parent[find(i)] = find(j)

    groups = {}
    for i in standing_idx:
        groups.setdefault(find(i), []).append(i)

    for d in detections:
        d["in_queue"] = False
        d["queue_group"] = None
        d["queue_reject"] = None
        d["rescued"] = False

    n_groups, rejected = 0, []
    for members in groups.values():
        size = len(members)
        pts = [foot_point(detections[i]["bbox"]) for i in members]
        reason = None

        if queue_zone is not None:
            frac_inside = sum(point_in_polygon(p, queue_zone) for p in pts) / size
            if frac_inside < zone_min_fraction:
                reason = "outside_zone"
            elif size < min_group_size_in_zone:
                reason = "too_small"
        else:
            if size < min_group_size:
                reason = "too_small"
            elif min_elongation is not None and size >= 3 and group_elongation(pts) < min_elongation:
                reason = "blob"

        if reason is None and min_classifier_fraction is not None:
            votes = sum(1 for i in members if detections[i].get("queue_vote", False))
            if votes / size < min_classifier_fraction:
                reason = "classifier"

        if reason is None:
            for i in members:
                detections[i]["in_queue"] = True
                detections[i]["queue_group"] = n_groups
            n_groups += 1
        else:
            for i in members:
                detections[i]["queue_reject"] = reason
            if size >= 2:
                rejected.append({"size": size, "reason": reason})

    if rescue_max_p_seated is not None:
        cands = [i for i, d in enumerate(detections)
                 if d.get("seated", False) and not d["in_queue"]
                 and d.get("p_seated") is not None and d["p_seated"] <= rescue_max_p_seated
                 and (min_member_conf is None or d.get("confidence", 1.0) >= min_member_conf)]
        changed = True
        while changed and cands:            # loop so a rescued person can anchor the next one in the line
            changed = False
            for i in list(cands):
                c = detections[i]
                if queue_zone is not None and not point_in_polygon(foot_point(c["bbox"]), queue_zone):
                    continue
                for j, m in enumerate(detections):
                    if j == i or not m["in_queue"]:
                        continue
                    if pair_distance(c, m, normalize) > dist_threshold:
                        continue
                    h = max(box_height(c["bbox"]), box_height(m["bbox"]))
                    if abs(c["bbox"][1] - m["bbox"][1]) <= rescue_head_tol * h:   # heads level with the line
                        c["in_queue"], c["queue_group"], c["rescued"] = True, m["queue_group"], True
                        cands.remove(i)
                        changed = True
                        break

    return n_groups, rejected

print("Distance-queue logic ready")

In [ ]:

# ======================================================================
# CELL 7d  (REPLACE)  - pipeline + settings
# ======================================================================
DEFAULT_CONF = 0.15
DEFAULT_IMGSZ = 1280                # detection resolution. Before this, imgsz was never set, so YOLO silently used 640

# --- queue settings: edit these (or override per call), no need to re-run this cell ---
QUEUE_DIST_THRESHOLD = 1.6          # link people closer than this many person-heights
QUEUE_MIN_GROUP_SIZE = 3            # people needed to call it a queue (outside a zone)
QUEUE_MIN_ELONGATION = 2.0          # line-shape check outside a zone; None = off
QUEUE_ZONE = None                   # e.g. [(x1,y1),(x2,y2),...] pixel polygon around the queue area
QUEUE_ZONE_MIN_FRACTION = 0.5       # share of a group that must stand inside the zone
QUEUE_MIN_GROUP_SIZE_IN_ZONE = 2    # a pair inside the zone counts as a queue
QUEUE_CLASSIFIER_FRACTION = None    # e.g. 0.5 = also need half the group to look "queue" to your classifier
SEATED_THRESHOLD = 0.5              # p_seated at/above this = "seated"
QUEUE_MIN_MEMBER_CONF = None        # e.g. 0.3 = boxes below this confidence can't join a queue (still counted as people)
QUEUE_RESCUE_MAX_P_SEATED = 0.9     # unsure "seated" people standing IN the line join it. None = off
QUEUE_RESCUE_HEAD_TOL = 0.2         # head height must match the line within this fraction of body height

def current_queue_cfg():
    return dict(
        dist_threshold=QUEUE_DIST_THRESHOLD,
        min_group_size=QUEUE_MIN_GROUP_SIZE,
        min_elongation=QUEUE_MIN_ELONGATION,
        queue_zone=QUEUE_ZONE,
        zone_min_fraction=QUEUE_ZONE_MIN_FRACTION,
        min_group_size_in_zone=QUEUE_MIN_GROUP_SIZE_IN_ZONE,
        min_classifier_fraction=QUEUE_CLASSIFIER_FRACTION,
        min_member_conf=QUEUE_MIN_MEMBER_CONF,
        rescue_max_p_seated=QUEUE_RESCUE_MAX_P_SEATED,
        rescue_head_tol=QUEUE_RESCUE_HEAD_TOL,
    )

def run_pipeline(image_path, conf=None, queue_mode="distance", imgsz=None, **overrides):
    conf = DEFAULT_CONF if conf is None else conf
    imgsz = DEFAULT_IMGSZ if imgsz is None else imgsz
    cfg = current_queue_cfg()
    unknown = set(overrides) - set(cfg)
    if unknown:
        raise TypeError(f"Unknown setting(s): {sorted(unknown)}. Valid: {sorted(cfg)}")
    cfg.update(overrides)

    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Could not read image (bad path or not a real JPEG/PNG): {image_path}")

    results = detection_model(image_path, conf=conf, iou=0.5, imgsz=imgsz,
                              classes=[0], agnostic_nms=True)[0]
    detections = [{"bbox": b.xyxy[0].tolist(), "confidence": round(float(b.conf[0]), 3)} for b in results.boxes]

    # step 1: seated vs standing (your trained classifier)
    for det in detections:
        x1, y1, x2, y2 = map(int, det["bbox"])
        crop = img[max(0, y1):y2, max(0, x1):x2]
        if crop.size == 0:
            det["seated"], det["p_seated"], det["_crop"] = False, None, None
            continue
        p = seated_probability(crop)
        det["p_seated"] = round(p, 3)
        det["seated"] = p >= SEATED_THRESHOLD
        det["_crop"] = crop

    # step 2: queue
    n_queue_groups, rejected = 0, []
    if queue_mode == "distance":
        if cfg["min_classifier_fraction"] is not None:
            for det in detections:      # per-person vote from your trained queue classifier
                det["queue_vote"] = (not det["seated"]) and det["_crop"] is not None and \
                    classify_crop(det["_crop"], queue_model_loaded, ["not_queue", "queue"]) == "queue"
        n_queue_groups, rejected = assign_queue_by_distance(detections, **cfg)
    elif queue_mode == "classifier":
        for det in detections:
            det["in_queue"] = det["_crop"] is not None and \
                classify_crop(det["_crop"], queue_model_loaded, ["not_queue", "queue"]) == "queue"
            det["queue_group"], det["queue_reject"] = None, None
    else:
        raise ValueError("queue_mode must be 'distance' or 'classifier'")

    for det in detections:
        det.pop("_crop", None)

    queue_count = sum(1 for d in detections if d["in_queue"])
    seated_count = sum(1 for d in detections if d["seated"] and not d["in_queue"])
    total = len(detections)
    crowd = compute_crowd_score(queue_count, seated_count, total)
    return {
        "headcount": total,
        "queue_count": queue_count,
        "queue_groups": n_queue_groups,
        "rejected_groups": rejected,
        "seated_count": seated_count,
        "standing_other_count": total - queue_count - seated_count,
        "rescued_count": sum(1 for d in detections if d.get("rescued")),
        "settings": cfg,
        "conf": conf,
        "imgsz": imgsz,
        **crowd,
        "detections": detections,
    }

print("run_pipeline ready")


In [ ]:

# ======================================================================
# CELL 8   (REPLACE)  - summary + picture (dashed red = rescued, number = p_seated)
# ======================================================================
def print_summary(result, image_name=""):
    print(f"--- {image_name} ---")
    print(f"Total people:   {result['headcount']}")
    rescued = f", incl. {result['rescued_count']} rescued (looked seated, but stand in the line)" if result.get("rescued_count") else ""
    print(f"In queue:       {result['queue_count']}  ({result['queue_groups']} line(s)){rescued}")
    print(f"Seated:         {result['seated_count']}")
    print(f"Standing/other: {result['standing_other_count']}")
    if result["rejected_groups"]:
        why = ", ".join(f"{g['size']} people ({g['reason']})" for g in result["rejected_groups"])
        print(f"Not a queue:    {why}")
    print(f"Crowd level:    {result['level'].upper()}  (score={result['score']})")


def show_result(image_path, result):
    level_colors = {"green": "limegreen", "amber": "orange", "red": "red"}
    cfg = result["settings"]
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(figsize=(12, 9))
    ax.imshow(img)

    # queue zone outline, if one is set
    if cfg["queue_zone"] is not None:
        ax.add_patch(patches.Polygon(cfg["queue_zone"], closed=True, fill=False,
                                     edgecolor="lime", linestyle="--", linewidth=2))

    dets = result["detections"]
    for det in dets:
        x1, y1, x2, y2 = det["bbox"]
        if det["in_queue"]:
            color = "red"
        elif det["seated"]:
            color = "cyan"
        elif det.get("queue_reject"):
            color = "magenta"          # close to someone, but filtered out as NOT a queue
        else:
            color = "orange"
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color,
                                       facecolor="none", linestyle="--" if det.get("rescued") else "-"))
        if det.get("p_seated") is not None and (det.get("rescued") or det["seated"]):
            ax.text(x1, y1 - 4, f"{det['p_seated']:.2f}", color=color, fontsize=9, weight="bold",
                    backgroundcolor="black")      # how sure the classifier was that this person is seated

    # yellow lines between people linked into the same queue group
    for a in range(len(dets)):
        for b in range(a + 1, len(dets)):
            if (dets[a]["in_queue"] and dets[b]["in_queue"]
                    and dets[a]["queue_group"] == dets[b]["queue_group"]
                    and pair_distance(dets[a], dets[b]) <= cfg["dist_threshold"]):
                pa, pb = foot_point(dets[a]["bbox"]), foot_point(dets[b]["bbox"])
                ax.plot([pa[0], pb[0]], [pa[1], pb[1]], color="yellow", linewidth=2)

    ax.set_title(f"CROWD: {result['level'].upper()} ({result['score']})",
                 color=level_colors[result["level"]], fontweight="bold")
    ax.legend(handles=[
        patches.Patch(color="red", label="In queue (dashed = rescued)"),
        patches.Patch(color="cyan", label="Seated"),
        patches.Patch(color="orange", label="Standing / other"),
        patches.Patch(color="magenta", label="Close together but NOT a queue"),
        plt.Line2D([0], [0], color="yellow", label="Linked (within threshold)"),
    ], loc="upper right", fontsize=9)
    ax.axis("off")
    plt.show()


test_image = f"{BASE}/mess_photos/IMG_20261005_200423.jpg"
result = run_pipeline(test_image)
print_summary(result, os.path.basename(test_image))
show_result(test_image, result)

In [ ]:


def show_seated_probs(image_path, conf=None, imgsz=None):
    conf = DEFAULT_CONF if conf is None else conf
    imgsz = DEFAULT_IMGSZ if imgsz is None else imgsz
    img = cv2.imread(image_path)
    res = detection_model(image_path, conf=conf, iou=0.5, imgsz=imgsz, classes=[0], agnostic_nms=True)[0]
    fig, ax = plt.subplots(figsize=(12, 9))
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    for b in res.boxes:
        x1, y1, x2, y2 = map(int, b.xyxy[0].tolist())
        crop = img[max(0, y1):y2, max(0, x1):x2]
        if crop.size == 0:
            continue
        t = _transform(Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))).unsqueeze(0).to(device)
        with torch.no_grad():
            p_seated = torch.softmax(seated_model_loaded(t), dim=1)[0][0].item()  # index 0 = "seated"
        color = "cyan" if p_seated >= 0.5 else "orange"
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor=color, facecolor="none"))
        ax.text(x1, y1 - 4, f"{p_seated:.2f}", color=color, fontsize=11, weight="bold", backgroundcolor="black")
    ax.axis("off")
    plt.show()

show_seated_probs(test_image)

In [ ]:

# ======================================================================
# CELL 10   (REPLACE or NEW, after Cell 8)  - tuner for your hand counts
# ======================================================================
import copy, itertools

PHOTO_DIR = f"{BASE}/mess_photos"

# ------------------------------------------------------------------
# 1) YOUR HAND COUNTS. For each photo, count by eye:
#      total  = everyone you can see
#      queue  = people actually waiting in line for food
#      seated = people sitting (standing/other is worked out for you)
#    Use 6+ photos that were NOT used to make your training crops.
# ------------------------------------------------------------------
LABELS = {
    # "IMG_1087-LOHITAKSHAROHILA.jpg": {"total": 27, "queue": 8, "seated": 16},
    # "photo2.jpg":                    {"total": 31, "queue": 0, "seated": 25},
}

# 2) What to try. Detection is run once per (photo, conf, imgsz); grouping is re-done instantly.
GRID = {
    "conf":           [0.10, 0.15, 0.25],
    "imgsz":          [1280],               # add 1920 if you have a GPU
    "dist_threshold": [1.0, 1.3, 1.6, 2.0],
    "min_group_size": [2, 3, 4],
    "min_elongation": [None, 1.5, 2.0],
    "rescue_max_p_seated": [None, 0.85, 0.95],   # None = rescue off. Any current_queue_cfg() key can be added here.
}
WEIGHTS = {"total": 1.0, "queue": 2.0, "seated": 1.0}   # queue counts double: it is the hard part
HOLDOUT_EVERY = 3                                       # every 3rd photo is kept aside to check for luck


def _counts(dets):
    return {"total": len(dets),
            "queue": sum(d["in_queue"] for d in dets),
            "seated": sum(d["seated"] and not d["in_queue"] for d in dets)}

def _image_error(pred, truth):
    # relative error per metric; denominator floored at 5 so a photo with 0 queue doesn't blow up the score
    return sum(w * abs(pred[k] - truth[k]) / max(truth[k], 5) for k, w in WEIGHTS.items()) / sum(WEIGHTS.values())

def _evaluate(cache, names, conf, imgsz, cfg):
    errs, abs_err = [], {k: 0.0 for k in WEIGHTS}
    for n in names:
        dets = copy.deepcopy(cache[(n, conf, imgsz)])
        assign_queue_by_distance(dets, **cfg)
        pred = _counts(dets)
        errs.append(_image_error(pred, LABELS[n]))
        for k in WEIGHTS:
            abs_err[k] += abs(pred[k] - LABELS[n][k]) / len(names)
    return sum(errs) / len(errs), abs_err

def _fmt_mae(m): return f"total {m['total']:.1f} | queue {m['queue']:.1f} | seated {m['seated']:.1f}"

def tune():
    if len(LABELS) < 3:
        raise ValueError("Fill LABELS with at least 3 photos (6+ recommended).")
    names = sorted(LABELS)
    holdout = names[::HOLDOUT_EVERY] if (HOLDOUT_EVERY and len(names) >= 6) else []
    tune_names = [n for n in names if n not in holdout]
    if not holdout:
        print("NOTE: fewer than 6 photos, so no holdout. Treat the result as a hint, not proof.\n")

    base_cfg, base_conf, base_imgsz = current_queue_cfg(), DEFAULT_CONF, DEFAULT_IMGSZ
    confs = sorted(set(GRID["conf"]) | {base_conf})
    imgszs = sorted(set(GRID["imgsz"]) | {base_imgsz})

    cache = {}
    for n in names:
        for c in confs:
            for s in imgszs:
                cache[(n, c, s)] = run_pipeline(f"{PHOTO_DIR}/{n}", conf=c, imgsz=s)["detections"]

    cfg_keys = [k for k in GRID if k not in ("conf", "imgsz")]
    rows = []
    for conf, imgsz in itertools.product(GRID["conf"], GRID["imgsz"]):
        for vals in itertools.product(*(GRID[k] for k in cfg_keys)):
            cfg = {**base_cfg, **dict(zip(cfg_keys, vals))}
            score, _ = _evaluate(cache, tune_names, conf, imgsz, cfg)
            rows.append((score, conf, imgsz, cfg))
    # Ties are common with few photos: the data can't tell those settings apart. Break them toward the
    # safer choice: rescue off, lowest link distance, shape check ON, min group size nearest the default of 3.
    rows.sort(key=lambda r: (round(r[0], 6), r[3]["rescue_max_p_seated"] is not None, r[3]["dist_threshold"],
                             r[3]["min_elongation"] is None, abs(r[3]["min_group_size"] - 3)))
    best_score = round(rows[0][0], 6)
    n_tied = sum(1 for r in rows if round(r[0], 6) == best_score)

    b_score, b_mae = _evaluate(cache, tune_names, base_conf, base_imgsz, base_cfg)
    print(f"CURRENT settings  | score {b_score:.3f} | avg miss per photo: {_fmt_mae(b_mae)}")
    if holdout:
        h_score, h_mae = _evaluate(cache, holdout, base_conf, base_imgsz, base_cfg)
        print(f"   on holdout     | score {h_score:.3f} | {_fmt_mae(h_mae)}")

    print(f"\nTOP 5 (lower score = better). {n_tied} setting(s) tie for the best score.")
    if n_tied > 5:
        print("  Many ties = your photos can't separate them. More hand-counted photos (esp. ones with a queue) will.")
    for score, conf, imgsz, cfg in rows[:5]:
        line = (f"  score {score:.3f} | conf={conf} imgsz={imgsz} dist={cfg['dist_threshold']} "
                f"min_size={cfg['min_group_size']} elong={cfg['min_elongation']} rescue={cfg['rescue_max_p_seated']}")
        if holdout:
            hs, _ = _evaluate(cache, holdout, conf, imgsz, cfg)
            line += f"  -> holdout {hs:.3f}"
        print(line)

    score, conf, imgsz, cfg = rows[0]
    print("\nPer-photo check with the best setting (predicted / hand count):")
    for n in names:
        dets = copy.deepcopy(cache[(n, conf, imgsz)])
        assign_queue_by_distance(dets, **cfg)
        p, t = _counts(dets), LABELS[n]
        tag = "  (holdout)" if n in holdout else ""
        print(f"  {n[:34]:<34} total {p['total']}/{t['total']}  queue {p['queue']}/{t['queue']}  seated {p['seated']}/{t['seated']}{tag}")

    print("\nTo use it, put these in a cell after 7d:")
    print(f"DEFAULT_CONF = {conf}\nDEFAULT_IMGSZ = {imgsz}\nQUEUE_DIST_THRESHOLD = {cfg['dist_threshold']}\n"
          f"QUEUE_MIN_GROUP_SIZE = {cfg['min_group_size']}\nQUEUE_MIN_ELONGATION = {cfg['min_elongation']}\n"
          f"QUEUE_RESCUE_MAX_P_SEATED = {cfg['rescue_max_p_seated']}")
    return rows[0]

# tune()      # <- run this once LABELS is filled in


In [ ]:

# ============================================================
# CELL 11 (NEW) — Batch processing + result logging.
# Run the whole pipeline over every photo in a folder at once, and
# save results to a JSON log so you have a persistent record across
# sessions instead of only seeing numbers scroll past in the output.
# ============================================================
def run_pipeline_batch(image_dir, log_path=None):
    image_paths = (
        glob.glob(f"{image_dir}/*.jpg")
        + glob.glob(f"{image_dir}/*.jpeg")
        + glob.glob(f"{image_dir}/*.png")
    )
    all_results = {}
    for path in image_paths:
        name = os.path.basename(path)
        result = run_pipeline(path)
        result.pop("detections")  # drop raw boxes from the saved log, keep it small
        all_results[name] = result
        print_summary(result, name)
        print()

    if log_path:
        existing = {}
        if os.path.exists(log_path):
            with open(log_path) as f:
                existing = json.load(f)
        existing[datetime.datetime.now().isoformat()] = all_results
        with open(log_path, "w") as f:
            json.dump(existing, f, indent=2)
        print(f"Logged results to {log_path}")

    return all_results


# Usage — runs every photo in mess_photos/, logs results with a timestamp
# each run, so you can track results over time / compare runs later.
log_path = f"{BASE}/pipeline_results_log.json"
batch_results = run_pipeline_batch(f"{BASE}/mess_photos", log_path=log_path)

In [ ]:
from google.colab import files
files.download(f"{BASE}/queue_classifier.pt")
files.download(f"{BASE}/seated_classifier.pt")